## Calculate Direct Converted Land by Technology and State

### Overview

This notebook calculates the direct converted land of wind, solar, and other technologies under a range of direct land use percentage assumptions. A table with these values is provided in the supplemental information of the manuscript.

### Imports

In [6]:
import pandas as pd
import numpy as np
import geopandas as gpd

import os

### Data Paths

In [7]:
# data dir
data_dir = os.path.join(os.path.dirname(os.getcwd()), 'data', 'input_data')

# infrastructure siting output csv
infrastucture_path = os.path.join(data_dir, 'infrastructure_data_csv', 'infrastructure_data.csv')

#### Prep Data

In [8]:
# read in infrastructure data
df_km = pd.read_csv(infrastucture_path)

# select cerf sitings only
df_km = df_km[df_km.cerf_sited==1]

# land usage aggregate by scenario, state and technology
df_km = df_km[['scenario', 'region_name', 'technology_simple', 'unit_size_mw']].groupby(['scenario','region_name', 
                                                                                          'technology_simple'], as_index=False).count()
df_km = df_km.rename(columns={'technology_simple':'Technology', 'region_name': 'State'})

df_km = pd.pivot_table(df_km, values='unit_size_mw', index=['State', 'scenario'], 
                      columns=['Technology'], aggfunc=np.sum).reset_index().rename_axis(None, axis=1)
df_km = df_km.fillna(0)
df_km = pd.melt(df_km, id_vars=['State', 'scenario'], value_vars=list(df_km.columns[2:]), var_name='Technology', value_name='value')
df_km['var'] = 'km-squared'


df_km['scenario'].replace('business_as_usual_ira_ccs_climate', 'Business-as-usual', inplace=True)
df_km['scenario'].replace('net_zero_ira_ccs_climate', 'Net Zero', inplace=True)
df_km['State'] = df_km['State'].str.title()
df_km['State'] = df_km['State'].replace("New_Mexico", "New Mexico")

df_km = df_km[['scenario', 'State', 'Technology', 'value']].sort_values(by=['scenario', 'State', 'Technology'])

#### Split Data by Technology Group

In [9]:
solar_data = df_km[df_km['Technology'].isin(['Solar PV'])].copy()
wind_data = df_km[df_km['Technology'].isin(['Wind', 'Offshore Wind'])].copy()
other_data = df_km[~df_km['Technology'].isin(['Wind', 'Offshore Wind', 'Solar PV'])].copy()

#### Calculate Direct Land Use

In [10]:
wind_data['low_value'] = wind_data['value'] * 0.02
wind_data['mid_value'] = wind_data['value'] * 0.035
wind_data['high_value'] = wind_data['value'] * 0.05

solar_data['low_value'] = solar_data['value'] * 0.60
solar_data['mid_value'] = solar_data['value'] * 0.75
solar_data['high_value'] = solar_data['value'] * 0.90


other_data['low_value'] = other_data['value'] * .8
other_data['mid_value'] = other_data['value'] * 0.9
other_data['high_value'] = other_data['value'] * 1

data = pd.concat([wind_data, solar_data, other_data])

In [11]:
data

,scenario,State,Technology,value,low_value,mid_value,high_value
66,Business-as-usual,Arizona,Offshore Wind,0.0,0.00,0.000,0.00
132,Business-as-usual,Arizona,Wind,614.0,12.28,21.490,30.70
68,Business-as-usual,California,Offshore Wind,240.0,4.80,8.400,12.00
134,Business-as-usual,California,Wind,2025.0,40.50,70.875,101.25
70,Business-as-usual,Colorado,Offshore Wind,0.0,0.00,0.000,0.00
...,...,...,...,...,...,...,...
107,Net Zero,Washington,Solar CSP,15.0,12.00,13.500,15.00
21,Net Zero,Wyoming,Biomass,18.0,14.40,16.200,18.00
43,Net Zero,Wyoming,Coal,1.0,0.80,0.900,1.00
65,Net Zero,Wyoming,Natural Gas,1.0,0.80,0.900,1.00
